In [ ]:
# THE CONFRONTATION CLIP - standalone video with the same upgraded dramatic voice + original score pipeline.
# Run top-to-bottom. It installs+starts ComfyUI, then pauses once at the cost meter before the final render.

In [ ]:
# STEP 0: put the kit at /kaggle/working/film_kit_v3 (robust to zip OR extracted folder)
import os, glob, zipfile, shutil
DEST = '/kaggle/working/film_kit_v3'
SENTINEL = 'config_confrontation.yaml'   # a file known to live at the kit root

def has_kit(d):
    return os.path.isfile(os.path.join(d, SENTINEL))

if not has_kit(DEST):
    roots = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/' + SENTINEL, recursive=True)]
    if roots:
        if os.path.isdir(DEST): shutil.rmtree(DEST)
        shutil.copytree(roots[0], DEST); print('copied kit from', roots[0])
    else:
        zips = glob.glob('/kaggle/input/**/*film_kit*.zip', recursive=True) \
             + glob.glob('/kaggle/working/*film_kit*.zip', recursive=True)
        assert zips, 'Add the film kit (zip or extracted folder) as a Kaggle Input first.'
        os.makedirs(DEST, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z: z.extractall(DEST)
        print('unzipped', zips[0], '->', DEST)
assert has_kit(DEST), DEST + ' missing (' + SENTINEL + ' not found)'
print('kit ready ->', DEST)

In [ ]:
import os
os.chdir('/kaggle/working/film_kit_v3')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['KIT_CONFIG'] = 'config_confrontation.yaml'

In [ ]:
# Use large MusicGen for the richest original score if memory allows; it falls back automatically if needed.
os.environ.setdefault('MG_MODEL', 'facebook/musicgen-large')
os.environ.setdefault('SCORE_CHUNK_SEC', '18')

In [ ]:
# STEP 0.5: install + launch headless ComfyUI at 127.0.0.1:8188 (render stages need it)
import os, subprocess, time, urllib.request

COMFY = '/kaggle/working/ComfyUI'
if not os.path.isdir(COMFY):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/comfyanonymous/ComfyUI', COMFY], check=True)
    subprocess.run(['pip', '-q', 'install', '-r', f'{COMFY}/requirements.txt'], check=True)

# custom nodes the workflows reference
CN = f'{COMFY}/custom_nodes'; os.makedirs(CN, exist_ok=True)
nodes = {
    'ComfyUI-GGUF':             'https://github.com/city96/ComfyUI-GGUF',
    'ComfyUI-WanVideoWrapper':  'https://github.com/kijai/ComfyUI-WanVideoWrapper',
    'ComfyUI_InstantID':        'https://github.com/cubiq/ComfyUI_InstantID',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite',
    'ComfyUI_UltimateSDUpscale':'https://github.com/ssitu/ComfyUI_UltimateSDUpscale',
}
for name, url in nodes.items():
    d = os.path.join(CN, name)
    if not os.path.isdir(d):
        subprocess.run(['git', 'clone', '--depth', '1', url, d])
        req = os.path.join(d, 'requirements.txt')
        if os.path.exists(req):
            subprocess.run(['pip', '-q', 'install', '-r', req])

# point ComfyUI at the mounted model dataset (see MANIFEST / build_model_dataset.ipynb)
if os.path.isdir('/kaggle/input/moment-models'):
    open(f'{COMFY}/extra_model_paths.yaml', 'w').write(
        'kaggle:\n  base_path: /kaggle/input/moment-models\n'
        '  checkpoints: .\n  diffusion_models: .\n  unet: .\n  clip: .\n  vae: .\n'
        '  clip_vision: .\n  loras: .\n  upscale_models: .\n  controlnet: .\n'
        '  instantid: .\n  insightface: .\n  audio_encoders: .\n')

def _up():
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/', timeout=3); return True
    except Exception:
        return False

if not _up():
    log = open('/kaggle/working/comfyui.log', 'w')
    subprocess.Popen(['python3', f'{COMFY}/main.py', '--listen', '127.0.0.1', '--port', '8188',
                      '--cache-none', '--disable-smart-memory'],
                     cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(120):
        if _up(): break
        time.sleep(3)
print('ComfyUI reachable:', _up(), '(if False: !tail -n 40 /kaggle/working/comfyui.log)')

In [ ]:
# STEP 0.6: smoke-test ComfyUI (server + required nodes + model visibility). No GPU render.
import json, urllib.request

def _api(path):
    return json.load(urllib.request.urlopen('http://127.0.0.1:8188' + path, timeout=15))

try:
    stats = _api('/system_stats')
    dev = (stats.get('devices') or [{}])[0].get('name', '?')
    print('server OK | device:', dev)
except Exception as e:
    raise SystemExit('ComfyUI not reachable - run STEP 0.5 first. ' + str(e))

info = _api('/object_info')
need = ['UnetLoaderGGUF','CLIPLoader','VAELoader','CLIPTextEncode','KSampler','VAEDecode',
        'WanSoundImageToVideo','WanPhantomSubjectToVideo','WanFirstLastFrameToVideo',
        'VHS_VideoCombine','CheckpointLoaderSimple','ApplyInstantID','ImageUpscaleWithModel']
missing = [n for n in need if n not in info]
print('required nodes present:', len(need) - len(missing), '/', len(need))
if missing:
    print('  MISSING nodes ->', missing)
    print('  (install the matching custom node, then re-run STEP 0.5)')

def _opts(cls, field):
    try:
        return info[cls]['input']['required'][field][0]
    except Exception:
        return []

for cls, field in [('UnetLoaderGGUF','unet_name'), ('VAELoader','vae_name'),
                   ('CLIPLoader','clip_name'), ('CheckpointLoaderSimple','ckpt_name')]:
    if cls in info:
        opts = _opts(cls, field)
        print(f'{cls}.{field}: {len(opts)} file(s)', opts[:4] if opts else '-- none found; check /kaggle/input/moment-models')
print()
print('Smoke test done. Fix any MISSING nodes / empty model lists before STEP 1.')

In [ ]:
# STEP 1: safe cost-meter run. Forges refs and renders/reuses ONE sample shot, then HOLDS.
os.environ.setdefault('BUDGET_USD', '4')
!python3 src/filmforge_confrontation.py

In [ ]:
# STEP 2: render the actual confrontation video after reviewing the cost meter.
# os.environ['CONFIRM_FULL_RUN'] = '1'
# !python3 src/filmforge_confrontation.py

In [ ]:
# STEP 3: package for download after STEP 2 finishes.
# !cd output_confrontation && zip -qr /kaggle/working/THE_CONFRONTATION_CLIP_final.zip THE_CONFRONTATION_CLIP.mp4 && echo 'Download from Output tab.'